# 02 — Paradigma 1: detección supervisada en CISUTAC (bloque A)

**Pregunta:** al detectar agujeros y manchas pequeños en fotos de prendas usadas, ¿conviene reducir la foto completa, aumentar la resolución de entrada o recorrerla con *tiles* a resolución nativa? ¿Cuánto cuesta cada opción?

El EDA mostró que, al reducir la foto (2160×3840) a 640 px, la mediana del defecto queda en **4,2 px (agujeros) y 3,0 px (manchas)**, y que el baseline preliminar a 640 no encontró ningún defecto menor a 32 px. Este notebook arma el **baseline supervisado** y lo compara de forma controlada contra variantes que conservan más detalle.

| ID | Variante | Entrenamiento | Inferencia | Qué responde |
|---|---|---|---|---|
| **A0** | Foto completa 640 | foto reducida a lado mayor 640 | ídem | Baseline estándar de YOLO |
| **A1** | Más resolución | foto reducida a lado mayor 1280 (opcional 1920) | ídem | ¿Alcanza con subir `imgsz`? |
| **A2** | Tiles sólo en inferencia | modelo de A0 | tiles 640 a resolución nativa | Ablación: ¿importa que la escala de entrenamiento y la de inferencia coincidan? |
| **A3** | Tiles en entrenamiento e inferencia | tiles 640 (20 % solapamiento) | ídem, con y sin la foto completa | Hipótesis principal |
| **A3-abl** | Ablaciones de A3 | proporción de tiles sin defectos; tamaño de tile | — | Qué decisiones de diseño importan |

**Reglas del protocolo** (`reports/protocolo_experimental.md`):

- Mismo detector (**YOLO11n** preentrenado en COCO), misma semilla, mismos aumentos y el mismo **presupuesto de entrenamiento** medido en imágenes vistas.
- Todas las métricas se calculan **en coordenadas de la foto original**, con `src/detection_eval.py`: mAP@0.5, mAP@0.5:0.95, AP_small, recall por tamaño, tiempo por imagen.
- **Valid** elige épocas, umbrales y variantes. **Test se usa una sola vez**, al final, para A0 y la mejor variante.
- Holes y Spots se tratan por separado: el notebook corre un dataset por vez (`DATASET`).

> **Cómo correrlo.** Necesita los datos de CISUTAC (ver `docs/DATASETS.md`), `pip install ultralytics==8.4.209 sahi==0.14.0 pycocotools==2.0.11` y, en lo posible, GPU. Primero conviene correrlo con `MODO_PRUEBA = True` (1 época, pocas imágenes, unos minutos) para verificar que todo funciona. Cada entrenamiento y evaluación queda guardado en `results/cisutac/bloque_a/`: si se vuelve a ejecutar el notebook, **se reutilizan** y no se reentrena (salvo `REENTRENAR = True`).

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import cisutac_baseline as cb
from detection_eval import SIZE_LABELS

pd.set_option("display.max_columns", 30)
print("Datos:", cb.DATA)
print("Resultados:", cb.RESULTS)

## 0. Parámetros

Todos los parámetros que se prueban están en esta celda o al comienzo de cada sección, para que se vea qué cambia entre variantes.

In [ ]:
DATASET = "holes"        # "holes" o "spots": correr el notebook una vez por dataset
MODO_PRUEBA = True       # True: 1 época y pocas imágenes, sólo para verificar el pipeline
REENTRENAR = False       # True: ignora corridas guardadas y vuelve a entrenar/evaluar
EVALUAR_TEST = False     # True sólo al final, una única vez, con las variantes ya elegidas en valid

# Comunes a todas las variantes (el único cambio entre A0, A1 y A3 es cómo se procesa la escala)
MODELO = "yolo11n.pt"    # preentrenado en COCO
SEMILLA = 0
EPOCAS_BASE = 100        # épocas para la foto completa; las demás variantes se igualan en imágenes vistas
PACIENCIA_BASE = 30      # épocas sin mejora en valid antes de cortar
CONF_EVAL = 0.001        # confianza mínima de las predicciones guardadas (bajo, para calcular mAP)

# Tiles: configuración principal (CIS-4: 97 % de los defectos entra entero en algún tile)
TILE, SOLAPAMIENTO = 640, 0.2
NEG_POR_POS = 1.0        # tiles sin defectos por cada tile con defectos
MIN_VISIBLE = 0.5        # fracción mínima de un defecto cortado para conservarlo como caja

if MODO_PRUEBA:
    EPOCAS_BASE, PACIENCIA_BASE = 1, 1
    FRACCION, LIMITE_EVAL, MAX_IMAGENES_TILES = 0.1, 3, 6
else:
    FRACCION, LIMITE_EVAL, MAX_IMAGENES_TILES = 1.0, None, None
SUFIJO = "_prueba" if MODO_PRUEBA else ""
BATCH = {640: 16, 1024: 8, 1280: 8, 1920: 4}   # se baja con la resolución para que entre en memoria

def nombre(variante):
    return f"{DATASET}_{variante}{SUFIJO}"

def pesos(run_dir):
    return Path(run_dir) / "weights" / "best.pt"

def evaluar_valid(run_name, weights, modo, **kw):
    # Evalúa en valid con la foto completa o con tiles, y nombra la evaluación según la configuración.
    detalle = (f"completa{kw.get('imgsz', 640)}" if modo == "completa" else
               f"tiles{kw.get('tile', TILE)}_o{round(100 * kw.get('overlap', SOLAPAMIENTO))}"
               + ("+completa" if kw.get("also_full") else ""))
    return cb.evaluate(f"{run_name}__valid_{detalle}", DATASET, weights, split="valid", mode=modo,
                       conf=CONF_EVAL, limit=LIMITE_EVAL, reevaluate=REENTRENAR, **kw)

In [ ]:
import importlib.metadata as md
for paquete in ("ultralytics", "sahi", "pycocotools", "torch"):
    try:
        print(f"{paquete:12s} {md.version(paquete)}")
    except md.PackageNotFoundError:
        print(f"{paquete:12s} FALTA -> pip install {paquete}")
print("Dispositivo:", cb.pick_device())

## 1. Datos

Se usan sólo las **fotos originales** con el split CISUTAC v1 (agrupado por día, `reports/cisutac/cis5_split_propuesto.csv`). Si todavía no existe `data/derived/`, la celda siguiente corre `scripts/prepare_cisutac_yolo.py`, que arma enlaces/copias, labels YOLO y las cajas COCO de referencia sin tocar los originales.

In [ ]:
if not (cb.derived_dir(DATASET) / "data.yaml").exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "prepare_cisutac_yolo.py")], check=True)

gt = {split: cb.load_ground_truth(DATASET, split) for split in ("train", "valid", "test")}
pd.DataFrame({split: {"imágenes": len(g["images"]), "defectos": len(g["annotations"]),
                      "defectos por imagen": round(len(g["annotations"]) / len(g["images"]), 2)}
              for split, g in gt.items()}).T

Distribución de tamaños por split (lado equivalente √(ancho·alto) en px de la original). Los rangos se fijaron en el EDA, antes de entrenar. Interesa que train y valid tengan suficientes defectos chicos para poder medir la diferencia entre variantes.

In [ ]:
from eda_utils import SIZE_EDGES

def rango(area):
    return SIZE_LABELS[sum(area ** 0.5 >= e for e in SIZE_EDGES)]

tamanos = pd.DataFrame({split: pd.Series([rango(a["area"]) for a in g["annotations"]]).value_counts()
                        for split, g in gt.items()}).reindex(SIZE_LABELS).fillna(0).astype(int)
tamanos.loc["COCO small (< 32 px)"] = [sum(a["area"] < 32 ** 2 for a in g["annotations"]) for g in gt.values()]
tamanos

**Por qué la escala importa.** Cada columna es un defecto chico real de valid: arriba a resolución nativa, abajo cómo queda cuando la foto completa se reduce a lado mayor 640 y 1280 (lo que recibe el modelo en A0 y A1).

In [ ]:
cb.plot_resolution_zoom(DATASET, sides=(640, 1280), n=4);

## 2. A0 — Baseline: foto completa a 640

YOLO11n preentrenado en COCO, ajustado con la foto completa reducida a lado mayor 640 (el `imgsz` por defecto de Ultralytics), con los aumentos por defecto (mosaico, escala ±50 %, giro horizontal, color). Valid sólo elige la mejor época.

In [ ]:
run_A0 = cb.train(nombre("A0_completa640"), cb.derived_dir(DATASET) / "data.yaml", imgsz=640,
                  epochs=EPOCAS_BASE, patience=PACIENCIA_BASE, batch=BATCH[640], seed=SEMILLA,
                  model=MODELO, retrain=REENTRENAR, fraction=FRACCION)
cb.plot_curves({"A0": run_A0});

Evaluación en valid, en coordenadas de la original. El umbral de confianza para precisión/recall/F1 es el que maximiza F1 en valid; mAP y AP_small no dependen de ese umbral.

In [ ]:
eval_A0 = evaluar_valid(run_A0.name, pesos(run_A0), "completa", imgsz=640)
display(cb.metrics_table([eval_A0]))
display(cb.size_table([eval_A0]))

In [ ]:
cb.plot_examples(DATASET, eval_A0["corrida"], n=4);

## 3. A1 — ¿Alcanza con subir la resolución de entrada?

Mismo modelo y receta, pero la foto completa entra con lado mayor más grande. Es el rival justo de los tiles: también procesa más píxeles, pero sin cortar la imagen. Se mantiene la misma cantidad de épocas que A0 (mismas imágenes vistas); el batch baja para que entre en memoria.

> 1920 necesita bastante memoria de GPU (≈ 9× los píxeles de 640). Agregarlo a la lista sólo si hay hardware.

In [ ]:
RESOLUCIONES_A1 = [1280]          # opcional: [1280, 1920]

runs_A1, evals_A1 = {}, []
for imgsz in RESOLUCIONES_A1:
    run = cb.train(nombre(f"A1_completa{imgsz}"), cb.derived_dir(DATASET) / "data.yaml", imgsz=imgsz,
                   epochs=EPOCAS_BASE, patience=PACIENCIA_BASE, batch=BATCH[imgsz], seed=SEMILLA,
                   model=MODELO, retrain=REENTRENAR, fraction=FRACCION)
    runs_A1[f"A1 {imgsz}"] = run
    evals_A1.append(evaluar_valid(run.name, pesos(run), "completa", imgsz=imgsz))

cb.plot_curves({"A0 640": run_A0, **runs_A1})
display(cb.metrics_table([eval_A0, *evals_A1]))
display(cb.size_table([eval_A0, *evals_A1]))

## 4. A2 — Tiles sólo en inferencia (ablación)

Se toma el modelo **ya entrenado en A0** y se lo aplica con tiles de 640 a resolución nativa (SAHI, fusión NMS a IoU 0,5), con y sin sumar la predicción de la foto completa. En el EDA esto **empeoró** el mAP: en los tiles los defectos se ven ~6 veces más grandes que en el entrenamiento. Se repite acá para que la comparación quede en la misma tabla.

In [ ]:
evals_A2 = [evaluar_valid(run_A0.name, pesos(run_A0), "tiles", tile=TILE, overlap=SOLAPAMIENTO, also_full=full)
            for full in (False, True)]
display(cb.metrics_table([eval_A0, *evals_A2]))
display(cb.size_table([eval_A0, *evals_A2]))

## 5. Dataset de tiles

Para entrenar a la misma escala con la que se infiere, cada foto de train y valid se recorre con la misma grilla que usa SAHI en inferencia. Cada tile se clasifica en:

- **con defecto:** contiene al menos un defecto que conserva `MIN_VISIBLE` (50 %) de su área. Las cajas cortadas se recortan al tile;
- **sin defecto:** no toca ningún defecto;
- **descartado:** sólo tiene fragmentos chicos de defectos cortados. Si se usaran como negativos, le enseñarían al modelo que medio agujero "no es defecto".

Como hay ~32 tiles por foto y la mayoría no tiene defectos, se toman `NEG_POR_POS` tiles sin defecto por cada tile con defecto (muestreo con semilla). Esa proporción es una decisión de diseño y se estudia en la sección 7.

**Presupuesto de entrenamiento.** El dataset de tiles tiene más imágenes que el de fotos completas, así que con las mismas épocas el modelo vería muchos más ejemplos. Para comparar en igualdad de condiciones se ajustan las épocas para ver aproximadamente la **misma cantidad de imágenes** que A0 (`cb.equivalent_epochs`).

In [ ]:
yaml_tiles, stats_tiles = cb.build_tile_dataset(DATASET, tile=TILE, overlap=SOLAPAMIENTO, neg_per_pos=NEG_POR_POS,
                                                seed=SEMILLA, min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES)
display(pd.DataFrame(stats_tiles).T)

n_completas = len(gt["train"]["images"]) if not MAX_IMAGENES_TILES else MAX_IMAGENES_TILES
def presupuesto(stats):
    # Épocas y paciencia equivalentes en imágenes vistas a EPOCAS_BASE sobre fotos completas.
    epocas = cb.equivalent_epochs(EPOCAS_BASE, n_completas, stats["train"]["tiles_totales"],
                                  minimum=1 if MODO_PRUEBA else 10)
    return epocas, max(1, round(PACIENCIA_BASE * epocas / EPOCAS_BASE))

EPOCAS_TILES, PACIENCIA_TILES = presupuesto(stats_tiles)
print(f"Épocas para tiles: {EPOCAS_TILES} (paciencia {PACIENCIA_TILES}) "
      f"≈ {EPOCAS_BASE} épocas × {n_completas} fotos completas")

In [ ]:
cb.plot_tile_samples(yaml_tiles, n=5);

## 6. A3 — Entrenar e inferir con tiles

Mismo YOLO11n, misma receta y presupuesto equivalente. En valid se infiere con SAHI sobre las **fotos originales completas** (no sobre los tiles), así que la evaluación es exactamente la misma que en A0–A2. Se prueba también sumar la predicción de la foto completa a 640, que en el EDA recuperaba defectos más grandes que un tile.

In [ ]:
def entrenar_tiles(variante, data_yaml, stats, tile=TILE):
    epocas, paciencia = presupuesto(stats)
    return cb.train(nombre(variante), data_yaml, imgsz=tile, epochs=epocas, patience=paciencia,
                    batch=BATCH[tile], seed=SEMILLA, model=MODELO, retrain=REENTRENAR)

run_A3 = entrenar_tiles(f"A3_tiles{TILE}_neg{NEG_POR_POS:g}", yaml_tiles, stats_tiles)
cb.plot_curves({"A3": run_A3})

evals_A3 = [evaluar_valid(run_A3.name, pesos(run_A3), "tiles", tile=TILE, overlap=SOLAPAMIENTO, also_full=full)
            for full in (False, True)]
display(cb.metrics_table([eval_A0, *evals_A1, *evals_A2, *evals_A3]))
display(cb.size_table([eval_A0, *evals_A1, *evals_A2, *evals_A3]))

In [ ]:
cb.plot_examples(DATASET, evals_A3[0]["corrida"], n=4);

## 7. Ablaciones de A3 (sólo en valid)

Se cambia **un factor por vez** a partir de la configuración principal de A3:

1. **Proporción de tiles sin defecto** (`NEG_POR_POS`). Con 0 el modelo nunca ve tela sana y se espera que confunda costuras, botones y estampados con defectos. Con muchos negativos aprende mejor el fondo, pero puede volverse conservador y perder recall.
2. **Tamaño de tile.** Tiles más chicos dan más contexto local y más pasadas (más costo); tiles más grandes, menos pasadas y más defectos enteros (CIS-4).

La inferencia usa siempre la grilla del mismo tamaño que el entrenamiento.

In [ ]:
ABLACION_NEGATIVOS = [0.0, 3.0]     # además del valor principal NEG_POR_POS

evals_neg, runs_neg = [], {}
for neg in ABLACION_NEGATIVOS:
    yaml_n, stats_n = cb.build_tile_dataset(DATASET, tile=TILE, overlap=SOLAPAMIENTO, neg_per_pos=neg, seed=SEMILLA,
                                            min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES)
    run = entrenar_tiles(f"A3_tiles{TILE}_neg{neg:g}", yaml_n, stats_n)
    runs_neg[f"neg {neg:g}"] = run
    evals_neg.append(evaluar_valid(run.name, pesos(run), "tiles", tile=TILE, overlap=SOLAPAMIENTO))

cb.plot_curves({f"neg {NEG_POR_POS:g}": run_A3, **runs_neg})
display(cb.metrics_table([evals_A3[0], *evals_neg]))
display(cb.size_table([evals_A3[0], *evals_neg]))

In [ ]:
ABLACION_TILES = [1024]             # opcional: [512, 1024]; 512 tiene ~2× el costo de inferencia de 640

evals_tile = []
for tile in ABLACION_TILES:
    yaml_t, stats_t = cb.build_tile_dataset(DATASET, tile=tile, overlap=SOLAPAMIENTO, neg_per_pos=NEG_POR_POS,
                                            seed=SEMILLA, min_visible=MIN_VISIBLE, max_images=MAX_IMAGENES_TILES)
    run = entrenar_tiles(f"A3_tiles{tile}_neg{NEG_POR_POS:g}", yaml_t, stats_t, tile=tile)
    evals_tile.append(evaluar_valid(run.name, pesos(run), "tiles", tile=tile, overlap=SOLAPAMIENTO))

display(cb.metrics_table([evals_A3[0], *evals_tile]))
display(cb.size_table([evals_A3[0], *evals_tile]))

## 8. Comparación de todas las variantes en valid

La tabla junta todas las evaluaciones de valid de este dataset que existan en `results/` (también las de corridas anteriores). Los gráficos muestran dónde ayuda cada variante: el recall por tamaño responde directamente la hipótesis (¿mejoran los defectos < 32 px?) y el gráfico de costo muestra a qué precio.

In [ ]:
todas = [json.loads(p.read_text()) for p in sorted((cb.RESULTS / "evaluaciones").glob(f"{DATASET}_*__valid_*/metricas.json"))]
todas = [m for m in todas if ("_prueba" in m["corrida"]) == MODO_PRUEBA]  # no mezclar pruebas con corridas reales
tabla_valid = cb.metrics_table(todas).sort_values("mAP@0.5", ascending=False)
tabla_valid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [2, 1]})
cb.plot_recall_by_size(todas, ax=axes[0])
cb.plot_cost_vs_quality(todas, metric="AP_small", ax=axes[1])
plt.tight_layout()
display(cb.size_table(todas))

## 9. Selección en valid y evaluación única en test

**Criterio fijado antes de mirar test:** se elige la variante con tiles de mayor **mAP@0.5 en valid** (métrica principal del protocolo); en caso de empate se prefiere la más barata. En test se evalúan sólo el baseline A0 y esa variante, cada una con **su umbral de confianza elegido en valid**.

Con un test de 64 (Holes) y 66 (Spots) fotos, si los intervalos de confianza se superponen **no se puede afirmar una mejora**.

In [ ]:
con_tiles = [m for m in todas if m["modo"] == "tiles" and "A3_" in m["corrida"]]
mejor = max(con_tiles, key=lambda m: (m["mAP@0.5"], -m["segundos_por_imagen"])) if con_tiles else None
print("Baseline :", eval_A0["corrida"])
print("Elegida  :", mejor and mejor["corrida"])

if EVALUAR_TEST and mejor:
    finales = []
    for m in (eval_A0, mejor):
        finales.append(cb.evaluate(m["corrida"].replace("__valid_", "__test_"), DATASET, m["pesos"], split="test",
                                   mode=m["modo"], imgsz=m["imgsz"], tile=m["tile"], overlap=m["solapamiento"],
                                   also_full=m["tambien_completa"], conf=CONF_EVAL, threshold=m["umbral_confianza"],
                                   confirm_test=True))
    display(cb.metrics_table(finales))
    display(cb.size_table(finales))
    cb.plot_recall_by_size(finales)
else:
    print("Test no evaluado (EVALUAR_TEST = False).")

## 10. Conclusiones

> Completar **después** de correr el notebook sin `MODO_PRUEBA`, con los números de las tablas. No escribir conclusiones antes de tener resultados.

- **A0 vs. A1:** ¿cuánto mejora AP_small y el recall < 32 px al subir la resolución de entrada? ¿Y a qué costo?
- **A2:** ¿se confirma que inferir con tiles un modelo entrenado con la foto reducida no sirve? ¿Por qué (escala, falsos positivos en ojalillos, costuras, estampados)?
- **A3 vs. A0/A1:** ¿los tiles mejoran los defectos chicos más que la resolución alta? ¿Mantienen los grandes? ¿Ayuda sumar la foto completa?
- **Ablaciones:** ¿qué proporción de negativos conviene y por qué? ¿Qué tamaño de tile da el mejor compromiso calidad/costo?
- **Holes vs. Spots:** ¿la conclusión es la misma para los dos tipos de defecto?
- **Limitaciones:** una semilla, valid chico, sin prendas sanas para medir falsos positivos en ropa sin defectos.

Este bloque es el **primer paradigma** del proyecto (supervisado, cajas). El segundo paradigma (detección de anomalías con PatchCore) y la comparación entre ambos en FabriD van en notebooks aparte.